# Train YOLO26x on Colab — crop-aligned subsets (`below_1000_crop`, `classification_crop`)

**Pick the best GPU first:** *Runtime → Change runtime type → GPU → **A100*** (Colab Pro+).
If A100 isn't available, **L4** > **V100** > **T4** (T4 works but is slow for yolo26x@1280).
Then **Runtime → Run all**.

This notebook trains the **crop-aligned** variants (train scale == inference scale → better for the
small/rare component & condition classes). Weights are saved back to Drive after each run.

In [ ]:
# @title 1) GPU check + auto-recommendation
import subprocess, torch
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout or "no nvidia-smi")
assert torch.cuda.is_available(), "No CUDA GPU! Runtime -> Change runtime type -> GPU (A100/L4)."
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory/1e9
print(f"GPU: {name} | VRAM: {vram:.0f} GB | torch {torch.__version__} CUDA {torch.version.cuda}")
tier = "A100" if "A100" in name else "L4" if "L4" in name else "V100" if "V100" in name else "T4" if "T4" in name else "other"
print("Tier:", tier, "(A100 best). yolo26x@1280 is heavy — A100/L4 strongly recommended.")

In [ ]:
# @title 2) Clone repo + install deps (Colab already has CUDA torch)
import os
REPO="/content/repo"
!git clone https://github.com/arupa444/trainDronisight.git $REPO 2>/dev/null || (cd $REPO && git pull)
%cd $REPO
!pip -q install uv && uv pip install --system -e .
import torch; print("CUDA still available:", torch.cuda.is_available())

In [ ]:
# @title 3) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title 4) Load datasets from Drive  ->  /content/data  (fast local SSD)
# EDIT `DRIVE_DATA` to where your crop subsets live, and `SUBSETS` to what you uploaded.
# Each subset may be a FOLDER or a .zip; this finds either. Looks in DRIVE_DATA and
# DRIVE_DATA/yolo_train_db.
import os, glob, zipfile, shutil
from pathlib import Path

DRIVE_DATA = "/content/drive/MyDrive/dronisight"     # <-- adjust if needed
SUBSETS    = ["component_below_1000_crop", "component_classification_crop"]
DEST       = "/content/data/yolo_train_db"
os.makedirs(DEST, exist_ok=True)

def locate(sub):
    cands = [f"{DRIVE_DATA}/yolo_train_db/{sub}", f"{DRIVE_DATA}/{sub}",
             f"{DRIVE_DATA}/yolo_train_db/{sub}.zip", f"{DRIVE_DATA}/{sub}.zip"]
    for c in cands:
        if os.path.exists(c): return c
    hits = glob.glob(f"{DRIVE_DATA}/**/{sub}*", recursive=True)
    return hits[0] if hits else None

for sub in SUBSETS:
    dst = f"{DEST}/{sub}"
    if os.path.isdir(dst) and glob.glob(f"{dst}/images/**/*.jpg", recursive=True):
        print("already present:", sub); continue
    src = locate(sub)
    assert src, f"could not find {sub} under {DRIVE_DATA} -- check the path/name"
    if src.endswith(".zip"):
        print("unzipping", src, "->", dst)
        with zipfile.ZipFile(src) as z: z.extractall(DEST)
        # if the zip contained a top-level <sub>/ keep it; else move
        if not os.path.isdir(dst):
            inner = glob.glob(f"{DEST}/**/{sub}", recursive=True)
            if inner: shutil.move(inner[0], dst)
    else:
        print("copying folder", src, "->", dst)
        shutil.copytree(src, dst, dirs_exist_ok=True)

os.environ["DRONISIGHT_DATA"] = "/content/data"      # config reads everything from here
# sanity: each subset resolves and has images
import importlib, shared.config as C; importlib.reload(C)
for sub in SUBSETS:
    p = C.YOLO_DB/sub
    n = len(glob.glob(f"{p}/images/train/clahe/*.jpg"))
    print(f"{sub}: {p} exists={p.is_dir()} train(clahe) imgs={n}")
    # drop any stale Ultralytics cache so it re-scans labels fresh
    for c in glob.glob(f"{p}/**/*.cache", recursive=True): os.remove(c)

In [ ]:
# @title 5) Train YOLO26x  (AMP on by default; batch=-1 = Ultralytics autobatch ~ uses the GPU fully)
# epochs: rare-class below_1000 gets more; condition gets the standard 150. Override BATCH to a
# fixed int (e.g. A100-40GB ~24-32, L4 ~10-14, T4 ~4-6) if you'd rather pin it than autobatch.
from train_yolo.train_components import run

BATCH = -1            # -1 = autobatch (fills the GPU); or set an int per the hints above
IMGSZ = 1280          # thin wires / small parts need high res
MODEL = "yolo26x.pt"  # falls back to yolo11x.pt with a warning if 26x isn't fetchable
EPOCHS = {"component_below_1000_crop": 200, "component_classification_crop": 150}

for sub in SUBSETS:
    print("="*70, f"\nTRAIN {sub}  epochs={EPOCHS.get(sub,150)} imgsz={IMGSZ} batch={BATCH} model={MODEL}\n", "="*70)
    run(subset=sub, version="clahe", epochs=EPOCHS.get(sub, 150), imgsz=IMGSZ, batch=BATCH, model=MODEL)
    # best weights: runs/<sub>/yolo/weights/best.pt

In [ ]:
# @title 6) Save runs/ (weights + plots) back to Drive  (Colab is ephemeral!)
from notebooks.colab_utils import save_runs_to_drive
print("saved to:", save_runs_to_drive())
# weights you'll use at inference:
import glob
for w in glob.glob("runs/*/yolo/weights/best.pt"): print(w)

In [ ]:
# @title 7) (optional) Peek final val metrics per run
import glob, pandas as pd
for csv in sorted(glob.glob("runs/*/yolo*/results.csv")):
    df = pd.read_csv(csv); df.columns = [c.strip() for c in df.columns]
    col = next((c for c in df.columns if "mAP50(B)" in c or "metrics/mAP50(B)" in c), None)
    if col is not None:
        print(f"{csv.split('/')[1]:34s} best val mAP@.5 = {df[col].max():.4f}  (epochs run: {len(df)})")